# Amphibian habitat and protected areas in Montérégie
**Core action:** amphibian conservation groups can see which recorded Western chorus frog habitats overlap a protected area, and which do not.

This notebook reads three datasets, prepares them, and writes the three GeoJSON files the web page loads:
`monteregie_amphibians.geojson`, `protected_monteregie.geojson` and `frog_protected_overlap.geojson`.
Run it top to bottom (Run All). The last cell is the check cell and should print `CHECK PASSED`.

Note: I switched from only looking at Western chorus frogs to all amphibians first, because the Western chorus frog data on GBIF was very outdated or scarce. The CDPNQ species-at-risk file has better Québec records, and the map now focuses on the chorus frog again.

In [58]:
import geopandas as gpd
import pyogrio

# Part 1: Amphibian records
**Step 1: Load the species data**

In [59]:
layers = pyogrio.list_layers("../data/emvs_dq.gpkg")
print(layers)

gdf = gpd.read_file("../data/emvs_dq.gpkg", layer="EMVS_publique")
print(f"All species records: {len(gdf)}")

[['EMVS_publique' 'MultiPolygon']]


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: Non-conformant content for record 1 in column EORANKDATE, 2023-11-02T00:00:00.0Z, successfully parsed
  return ogr_read(


All species records: 13298


**Step 2: Keep only amphibians** (matched on the scientific name, whole words only, so that plants like *Notothylas* are not matched by *Hyla*)

In [60]:
amphibian_genera = [
    "Rana", "Lithobates", "Pseudacris", "Hyla",
    "Anaxyrus", "Bufo",
    "Ambystoma", "Plethodon", "Desmognathus",
    "Notophthalmus", "Necturus", "Eurycea"
]
pattern = r"\b(?:" + "|".join(amphibian_genera) + r")\b"
amphibian_mask = gdf["SNAME"].str.contains(pattern, case=False, na=False, regex=True)
amphibians = gdf[amphibian_mask]
print(f"Records before: {len(gdf)}, amphibians kept: {len(amphibians)}, removed: {len(gdf) - len(amphibians)}")

Records before: 13298, amphibians kept: 856, removed: 12442


**Step 3: Convert the coordinates** from EPSG:32198 (NAD83 / Québec Lambert, metres) to EPSG:4326 (latitude and longitude, what Leaflet uses)

In [61]:
print("Converting from:", amphibians.crs)
amphibians_latlon = amphibians.to_crs(epsg=4326)
print("Converted to:", amphibians_latlon.crs)

Converting from: EPSG:32198
Converted to: EPSG:4326


# Part 2: The Montérégie boundary
**Step 1: Look at the layers and columns**

In [62]:
region_layers = pyogrio.list_layers("../data/SDA.gpkg")
print(region_layers)

regions = gpd.read_file("../data/SDA.gpkg", layer="regio_s")
print(regions.columns.tolist())

[['arron_s' 'MultiPolygon Z']
 ['comet_s' 'MultiPolygon Z']
 ['arron_l' 'LineString Z']
 ['comet_l' 'LineString Z']
 ['mrc_l' 'LineString Z']
 ['munic_l' 'LineString Z']
 ['regio_l' 'LineString Z']
 ['mrc_s' 'MultiPolygon Z']
 ['munic_s' 'MultiPolygon Z']
 ['regio_s' 'MultiPolygon Z']]
['RES_ID_IND', 'RES_NO_IND', 'RES_DE_IND', 'RES_CO_REG', 'RES_NM_REG', 'RES_CO_REF', 'RES_CO_VER', 'geometry']


**Step 2: Find Montérégie in the column `RES_NM_REG`**

In [63]:
monteregie = regions[regions["RES_NM_REG"].str.contains("Montérégie", case=False, na=False)]
print(f"Montérégie polygon(s) found: {len(monteregie)}")

Montérégie polygon(s) found: 1


**Step 3: Convert the boundary** (it is in EPSG:4269) to EPSG:4326

In [64]:
print("Converting from:", monteregie.crs)
monteregie_4326 = monteregie.to_crs(epsg=4326)
monteregie_boundary = monteregie_4326.union_all()
print("Boundary bounds:", monteregie_4326.total_bounds)

Converting from: EPSG:4269
Boundary bounds: [-74.66979689  44.99135972 -72.31555085  46.11346501]


# Part 3: Clip the amphibians to Montérégie
The study area is the real Montérégie boundary, not a rectangle.

In [65]:
monteregie_amphibians = amphibians_latlon[amphibians_latlon.within(monteregie_boundary)].copy()

print(f"Amphibians in Québec: {len(amphibians_latlon)}")
print(f"Amphibians inside Montérégie: {len(monteregie_amphibians)}")
print(monteregie_amphibians["SCOMNAME"].value_counts())

Amphibians in Québec: 856
Amphibians inside Montérégie: 90
SCOMNAME
Rainette faux-grillon de l'Ouest                        56
Salamandre sombre du Nord                               19
Salamandre sombre des montagnes, pop. des Appalaches    14
Grenouille des marais                                    1
Name: count, dtype: int64


# Part 4: Protected areas
**Step 1: Load the registry** (Registre des aires protégées et des AMCE, layer `AP_REG_S`)

In [66]:
print(pyogrio.list_layers("../data/registre_aires_prot_GPKG/registre_aires_prot.gpkg"))

protected = gpd.read_file("../data/registre_aires_prot_GPKG/registre_aires_prot.gpkg", layer="AP_REG_S")
print(protected.shape, protected.crs)
print(protected.columns.tolist())

[['AP_REG_S' 'MultiPolygon']
 ['AP_ZON_S' 'MultiPolygon']
 ['T_IMP_S' 'MultiPolygon']]
(5611, 19) EPSG:32198
['MACODE', 'TOPONYME', 'NOM', 'ARTICLE', 'DESIG_NO', 'DESIG_GR', 'DESIGNOM', 'RESPONSABL', 'CORESPONSA', 'HA_LEGAL', 'LIEN', 'LONDMS', 'LATDMS', 'DA_CREATIO', 'DURE_RECON', 'UICN', 'BIOME', 'note', 'geometry']


**Step 2: Convert and clip to Montérégie** (`intersects`, not `within`: a protected area can sit partly outside the region)

In [67]:
print("Converting from:", protected.crs)
protected_4326 = protected.to_crs(epsg=4326)
protected_monteregie = protected_4326[protected_4326.intersects(monteregie_boundary)].copy()

print(f"Protected areas in Québec: {len(protected_4326)}")
print(f"Protected areas touching Montérégie: {len(protected_monteregie)}")

Converting from: EPSG:32198
Protected areas in Québec: 5611
Protected areas touching Montérégie: 266


**Step 3: Keep only designations that protect general habitat.**
I remove areas designated for plants, birds or muskrats, because they are not aimed at amphibians.

In [68]:
n0 = len(protected_monteregie)

# plant habitat designations
protected_monteregie = protected_monteregie[
    ~protected_monteregie["DESIGNOM"].str.contains("floristique", case=False, na=False)
]
n1 = len(protected_monteregie)

# bird, muskrat and heron designations
protected_monteregie = protected_monteregie[
    ~protected_monteregie["DESIGNOM"].str.contains("oiseaux|rat musqu|Héronni", case=False, na=False)
].copy()
n2 = len(protected_monteregie)

print(f"Before: {n0}, after removing plant habitat: {n1}, after removing bird and muskrat habitat: {n2}")
print(protected_monteregie["DESIGNOM"].value_counts())

Before: 266, after removing plant habitat: 259, after removing bird and muskrat habitat: 189
DESIGNOM
Milieu naturel de conservation volontaire              120
Réserve naturelle reconnue                              55
Réserve écologique                                       5
Réserve nationale de faune                               3
Autres mesures de conservation efficaces                 2
Parc national                                            2
Autres mesures de conservation efficaces (fédérale)      1
Réserve de biodiversité projetée                         1
Name: count, dtype: int64


# Part 5: Which frog sites overlap a protected area?
**Step 1: Flag every amphibian record** with the protected areas it touches (`predicate="intersects"` means any overlap counts)

In [69]:
prot_small = protected_monteregie[["NOM", "geometry"]]
joined = gpd.sjoin(monteregie_amphibians, prot_small, how="left", predicate="intersects")

names = (joined[joined["NOM"].notna()]
         .groupby(level=0)["NOM"]
         .agg(lambda s: "; ".join(sorted(set(s)))))

monteregie_amphibians["protected_by"] = names
monteregie_amphibians["is_protected"] = monteregie_amphibians["protected_by"].notna()

print(monteregie_amphibians.groupby("SNAME")["is_protected"].agg(["sum", "count"]))

                                 sum  count
SNAME                                      
Desmognathus fuscus                2     19
Desmognathus ochrophaeus pop. 2    0     14
Lithobates palustris               0      1
Pseudacris triseriata             10     56


**Step 2: Cut out the overlap** (the part of each chorus frog habitat that lies inside a protected area)

# Part 6: Write the GeoJSON

In [70]:
# 1. amphibian records in Montérégie (all species; the page draws only the chorus frog)
monteregie_amphibians.to_file("../data/monteregie_amphibians.geojson", driver="GeoJSON")

# 2. protected areas (only the columns the popup uses)
protected_monteregie[["NOM", "DESIGNOM", "DA_CREATIO", "geometry"]].to_file(
    "../data/protected_monteregie.geojson", driver="GeoJSON")

# 3. chorus frog habitat inside a protected area
if len(overlap) > 0:
    overlap.to_file("../data/frog_protected_overlap.geojson", driver="GeoJSON")
else:
    print("No overlap found, so no overlap file was written.")

print(len(monteregie_amphibians), monteregie_amphibians.crs, monteregie_amphibians.total_bounds)

90 EPSG:4326 [-74.37793902  45.00624059 -72.82940126  45.82041049]


# Check cell
Tests the file the page loads: count, projection, area and properties. It must print `CHECK PASSED`.

In [71]:
import os

check = gpd.read_file("../data/monteregie_amphibians.geojson")
chorus = check[check["SNAME"] == "Pseudacris triseriata"]
print(len(check), "records;", len(chorus), "chorus frog records;", check.crs, check.total_bounds)

# count
assert len(check) == 90, "expected 90 amphibian records in Montérégie"
assert len(chorus) == 56, "expected 56 Western chorus frog records"

# projection
assert check.crs.to_epsg() == 4326, "the file should be in latitude and longitude (EPSG:4326)"

# area: every record must sit inside the Montérégie area
minx, miny, maxx, maxy = check.total_bounds
assert -74.6 < minx and maxx < -72.7 and 44.9 < miny and maxy < 46.0, "records outside the Montérégie area"

# properties the page uses
for col in ["GCOMNAME", "SNAME", "LOIEMV", "protected_by", "is_protected"]:
    assert col in check.columns, "missing column: " + col

# the headline and the overlap layer must agree
n_protected = int(chorus["is_protected"].astype(bool).sum())
overlap_file = os.path.exists("../data/frog_protected_overlap.geojson")
assert (n_protected > 0) == overlap_file, "is_protected and the overlap file disagree"

print(f"{n_protected} of {len(chorus)} chorus frog sites overlap a protected area")
print("CHECK PASSED")

90 records; 56 chorus frog records; EPSG:4326 [-74.37793902  45.00624059 -72.82940126  45.82041049]
10 of 56 chorus frog sites overlap a protected area
CHECK PASSED
